In [1]:
from pathlib import Path

import jupyter_black
import pandas as pd
from dotenv import load_dotenv

# Use verbose pandas mode
pd.options.display.max_rows = 200

load_dotenv()
jupyter_black.load()

## Data Loading and Initial Cleaning

In [2]:
data_path = Path("/workspace/data/llmPredictions")


# Load and clean
def load_and_clean(filepath):
    """
    Load a CSV file containing LLM-based SDG predictions and perform minimal cleaning.

    Parameters
    ----------
    filepath : Path
        Path to the CSV file containing predictions.

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with the following preprocessing steps applied:
            - Drop rows with missing DOIs.
            - Drop duplicate DOIs, keeping the first occurrence.

    Notes
    -----
    Each prediction file contains records retrieved by SDG Boolean queries.
    Duplicate DOIs may appear when multiple abstracts are merged across runs.
    The DOI field serves as a unique document identifier.
    """
    df = pd.read_csv(filepath)
    df = df.dropna(subset=["DOI"])  # remove rows with missing DOIs
    df = df.drop_duplicates(subset="DOI", keep="first")  # keep only one row per DOI
    return df


df_sdg1_qwen = load_and_clean(data_path / "scopus_sdg1_qwen_march12.csv")
df_sdg1_llama = load_and_clean(data_path / "scopus_sdg1_llama_3.1_march12.csv")
df_sdg3_qwen = load_and_clean(data_path / "scopus_sdg3_qwen.csv")
df_sdg3_llama = load_and_clean(data_path / "scopus-sdg3-llama3.1.csv")
df_sdg7_qwen = load_and_clean(data_path / "scopus_sdg7_qwen.csv")
df_sdg7_llama = load_and_clean(data_path / "scopus-sdg7-llama3.1.csv")

In [3]:
# Basic sanity check
print("SDG1 Qwen:", df_sdg1_qwen.shape)
print("SDG1 Llama:", df_sdg1_llama.shape)
print("SDG3 Qwen:", df_sdg3_qwen.shape)
print("SDG3 Llama:", df_sdg3_llama.shape)
print("SDG7 Qwen:", df_sdg7_qwen.shape)
print("SDG7 Llama:", df_sdg7_llama.shape)

SDG1 Qwen: (15133, 7)
SDG1 Llama: (15133, 6)
SDG3 Qwen: (15674, 7)
SDG3 Llama: (15674, 6)
SDG7 Qwen: (15948, 7)
SDG7 Llama: (15948, 6)


In [4]:
# Check total vs. unique DOI counts
total_rows = len(df_sdg1_qwen)
unique_dois = df_sdg1_qwen["DOI"].nunique()
print(f"Total rows: {total_rows}")
print(f"Unique DOIs: {unique_dois}")
print(f"Duplicate DOIs: {total_rows - unique_dois}")

# Show examples of duplicated DOIs
dup_dois = df_sdg1_qwen["DOI"][df_sdg1_qwen["DOI"].duplicated()].unique()
print(f"Number of duplicated DOIs: {len(dup_dois)}")

# Inspect a few of them
df_sdg1_qwen[df_sdg1_qwen["DOI"].isin(dup_dois)].sort_values("DOI").head(10)

Total rows: 15133
Unique DOIs: 15133
Duplicate DOIs: 0
Number of duplicated DOIs: 0


,Unnamed: 0.1,Unnamed: 0,DOI,Abstract,SDG,qwen-result,qwen_label


### Column Normalization and Dataset Consolidation

In [5]:
# Drop unnecessary columns
drop_cols = ["Unnamed: 0.1", "Unnamed: 0"]
for df in [
    df_sdg1_qwen,
    df_sdg1_llama,
    df_sdg3_qwen,
    df_sdg3_llama,
    df_sdg7_qwen,
    df_sdg7_llama,
]:
    df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

In [6]:
# Rename columns for consistency
rename_map = {
    "qwen-result": "qwen_result",
    "qwen_label": "model_label",
    "llama-3.1-result": "llama_result",
    "llama_label": "model_label",
}

for df in [
    df_sdg1_qwen,
    df_sdg1_llama,
    df_sdg3_qwen,
    df_sdg3_llama,
    df_sdg7_qwen,
    df_sdg7_llama,
]:
    df.rename(columns=rename_map, inplace=True)

In [7]:
# Add SDG & Model Identifiers
for df, sdg, model in [
    (df_sdg1_qwen, "SDG1", "Qwen"),
    (df_sdg1_llama, "SDG1", "Llama"),
    (df_sdg3_qwen, "SDG3", "Qwen"),
    (df_sdg3_llama, "SDG3", "Llama"),
    (df_sdg7_qwen, "SDG7", "Qwen"),
    (df_sdg7_llama, "SDG7", "Llama"),
]:
    df["SDG"] = sdg
    df["Model"] = model

**Purpose:**  
Explicit identifiers (SDG, Model) are added to each frame so that they can later be concatenated and grouped systematically. This enables cross-model, cross-SDG comparison and traceability back to the original classification context.

In [8]:
# Vaidiate the changes
print("SDG 1 - Qwen Columns:", df_sdg1_qwen.columns)
print("SDG 1 - Llama Columns:", df_sdg1_llama.columns)
print("SDG 3 - Qwen Columns:", df_sdg3_qwen.columns)
print("SDG 3 - Llama Columns:", df_sdg3_llama.columns)
print("SDG 7 - Qwen Columns:", df_sdg7_qwen.columns)
print("SDG 7 - Llama Columns:", df_sdg7_llama.columns)

SDG 1 - Qwen Columns: Index(['DOI', 'Abstract', 'SDG', 'qwen_result', 'model_label', 'Model'], dtype='object')
SDG 1 - Llama Columns: Index(['DOI', 'Abstract', 'SDG', 'llama_result', 'model_label', 'Model'], dtype='object')
SDG 3 - Qwen Columns: Index(['DOI', 'Abstract', 'SDG', 'qwen_result', 'model_label', 'Model'], dtype='object')
SDG 3 - Llama Columns: Index(['DOI', 'Abstract', 'SDG', 'llama_result', 'model_label', 'Model'], dtype='object')
SDG 7 - Qwen Columns: Index(['DOI', 'Abstract', 'SDG', 'qwen_result', 'model_label', 'Model'], dtype='object')
SDG 7 - Llama Columns: Index(['DOI', 'Abstract', 'SDG', 'llama_result', 'model_label', 'Model'], dtype='object')


## Merge all data into one dataframe

In [9]:
df_all = pd.concat(
    [
        df_sdg1_qwen,
        df_sdg1_llama,
        df_sdg3_qwen,
        df_sdg3_llama,
        df_sdg7_qwen,
        df_sdg7_llama,
    ],
    ignore_index=True,
)

# Remove rows with missing DOIs or Abstracts
df_all = df_all.dropna(subset=["DOI"])
df_all = df_all.dropna(subset=["Abstract"])

# Normalize DOIs
df_all["DOI"] = df_all["DOI"].str.strip().str.lower()

# Reorder columns for better readability
df_all = df_all[
    [
        "DOI",
        "Abstract",
        "SDG",
        "Model",
        "model_label",
    ]
]

In [10]:
# Check for (DOI, Model, SDG) duplicates
dupe_groups = (
    df_all.groupby(["DOI", "Model", "SDG"])["model_label"].nunique().reset_index()
)

# Filter for inconsistent labeling (same input, different outputs)
inconsistent_labeling = dupe_groups[dupe_groups["model_label"] > 1]

print(f"Inconsistent (DOI, Model, SDG) labelings: {len(inconsistent_labeling)}")

# Optional: inspect the conflicts
if not inconsistent_labeling.empty:
    conflicting_entries = df_all.merge(
        inconsistent_labeling.drop(columns=["model_label"]),
        on=["DOI", "Model", "SDG"],
        how="inner",
    ).sort_values(["DOI", "Model", "SDG"])
    conflicting_entries.to_csv("inconsistent_labeling_debug.csv", index=False)

Inconsistent (DOI, Model, SDG) labelings: 0


In [14]:
# Save merged dataset
output_path = data_path / "merged_sdgs_predictions.csv"
df_all.to_csv(output_path, index=False)
print(f"Merging complete. Dataset saved as {output_path}")

print(f"Rows: {len(df_all)}, Unique DOIs: {df_all['DOI'].nunique()}")
print(f"Saved on: {pd.Timestamp.now()}")

Merging complete. Dataset saved as /workspace/data/llmPredictions/merged_sdgs_predictions.csv
Rows: 93510, Unique DOIs: 46572
Saved on: 2025-11-10 22:12:02.477434
